In [66]:
import numpy as np

In [67]:
def softmax(x):
    x = np.array(x)
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / np.sum(e, axis=-1, keepdims=True)


In [68]:
def gradsoftmax(input):
    l=np.array(input)
    f,g=np.exp(l),np.sum(np.exp(l))
    return (f*g+f**2)/g**2

In [69]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=False)
list(iris)

['data',
 'target',
 'frame',
 'target_names',
 'DESCR',
 'feature_names',
 'filename',
 'data_module']

In [70]:
X=iris.data
y=iris.target

In [71]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42)

In [72]:
X_train[0]

array([5. , 3.6, 1.4, 0.2])

In [73]:
len(X_train)

112

In [74]:
W = np.random.randn(4, 3)   
X = np.array(X_train)     
b = np.random.randn(1,3)

logits = X @ W + b         
softmax1 = softmax(logits)


In [75]:
softmax1

array([[2.22140899e-04, 6.31387100e-01, 3.68390759e-01],
       [2.35714185e-04, 6.07914933e-01, 3.91849353e-01],
       [1.50403176e-05, 1.47630889e-01, 8.52354071e-01],
       [2.11667592e-05, 2.35590063e-01, 7.64388771e-01],
       [4.64933776e-06, 1.50746228e-01, 8.49249123e-01],
       [9.81499604e-05, 7.66137382e-01, 2.33764468e-01],
       [1.18636289e-04, 6.13518034e-01, 3.86363329e-01],
       [3.04403762e-05, 1.64233825e-01, 8.35735734e-01],
       [6.57504706e-06, 8.15775926e-02, 9.18415832e-01],
       [6.24485415e-06, 9.21682004e-02, 9.07825555e-01],
       [5.57484545e-06, 1.75332062e-01, 8.24662363e-01],
       [1.86952455e-06, 1.71054716e-01, 8.28943415e-01],
       [2.72723678e-05, 1.76368858e-01, 8.23603870e-01],
       [4.57792445e-07, 3.80327314e-02, 9.61966811e-01],
       [3.23456328e-05, 1.75445760e-01, 8.24521894e-01],
       [1.74825638e-04, 6.22601473e-01, 3.77223702e-01],
       [3.04712382e-07, 2.58174573e-02, 9.74182238e-01],
       [2.01605846e-05, 1.15688

In [76]:
print(iris.target_names[np.argmax(softmax1)%3])

virginica


In [77]:
def cross_entropy(X, y, W,b):
    loss = 0
    for i in range(len(X)):
        logits = X[i] @ W + b           
        probs = softmax(logits)        
        loss += np.sum(y[i] * np.log(probs + 1e-9))
    return -loss / len(X)


In [78]:
cross_entropy(X_train,y_train,W,b)

np.float64(14.472387537469682)

In [80]:
def grad_crossentropy(X,y,W,b):
    grad_w = np.zeros_like(W)
    grad_b = np.zeros_like(b)
    for i in range(len(X)):
        logits = X[i] @ W + b
        probs = softmax(logits)
        grad_w += np.outer(X[i],probs-y[i])
        grad_b += probs-y[i]
    return grad_w / len(X) , grad_b/len(X)


In [81]:
len(X_train)

112

In [83]:
W = np.random.randn(4, 3)     
b = np.random.randn(1,3)

X_subtrain,X_valid ,y_subtrain,y_valid  = X_train[:90] ,X_train[90:],y_train[:90],y_train[90:]
eta = 0.1
n_epoches = 100
best_valid_error = float('inf')
for epoch in range(n_epoches):
    W -= eta*grad_crossentropy(X_subtrain,y_subtrain,W,b)[0]
    b -= eta*grad_crossentropy(X_subtrain,y_subtrain,W,b)[1]
    val_error = cross_entropy(X_valid,y_valid,W,b)
    if val_error < best_valid_error:
        best_valid_error = val_error
    else:
        break
 


In [84]:
W,b

(array([[ 2.24930662,  1.05765815,  2.31316766],
        [ 0.94010614,  2.06426385, -1.27915299],
        [ 0.52248874,  1.27899688,  1.94357725],
        [-0.52644792,  0.58063312,  0.02339707]]),
 array([[ 0.90632501, -0.03070221,  0.18809001]]))

In [88]:
X_train[0]
softmax(X_train[0] @ W + b)

array([[0.82619651, 0.1722692 , 0.00153428]])

In [89]:
np.argmax(softmax(X_train[0] @ W + b))

np.int64(0)

In [90]:
y_train[0]

np.int64(0)

In [91]:
score = 0
for i in range(len(X_test)):
    if np.argmax(softmax(X_test[i] @ W + b)) == y_test[i]:
        score +=1

print(score/len(X_test))

0.7368421052631579
